## Setup

In [6]:
import os

# Define the base directory within Google Drive for your project data
BASE_DIR = '../data'
DATA_SUBFOLDER_NAME = 'N1_tupi_none_short_r1_778989'
FULL_DATA_PATH = os.path.join(BASE_DIR, DATA_SUBFOLDER_NAME)

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(FULL_DATA_PATH, exist_ok=True)

print(f"Permanent 'data' folder created at: {BASE_DIR}")
print(f"Permanent subfolder '{DATA_SUBFOLDER_NAME}' created at: {FULL_DATA_PATH}")

for folder in os.listdir(BASE_DIR):
    print(folder)

Permanent 'data' folder created at: ../data
Permanent subfolder 'N1_tupi_none_short_r1_778989' created at: ../data/N1_tupi_none_short_r1_778989
potiN2tupiN1
N1_tupi_none_short_r1_778989


In [7]:
DATA_SUBFOLDER_NAME_MULTI = 'potiN2tupiN1'
FULL_DATA_PATH_MULTI = os.path.join(BASE_DIR, DATA_SUBFOLDER_NAME_MULTI)

print(f"Permanent subfolder '{DATA_SUBFOLDER_NAME_MULTI}' path: {FULL_DATA_PATH_MULTI}")

for folder in os.listdir(FULL_DATA_PATH_MULTI):
    print(folder)

Permanent subfolder 'potiN2tupiN1' path: ../data/potiN2tupiN1
N2_poti_PP_long_r1_780376
N2_poti_TP_short_r1_780378
N2_poti_PP_short_r1_780413
N1_tupi_none_short_r1_780423
N2_poti_TP_long_r1_780377
N1_tupi_none_long_r1_780424


### Create table

In [8]:
import os
import json
import pandas as pd
import re

# ---------- CONFIG ----------
SINGLE_NAME = 'N1_tupi_none_short_r1_778989'
MULTI_NAME = 'potiN2tupiN1'

SINGLE_PATH = os.path.join(BASE_DIR, SINGLE_NAME)
MULTI_PATH = os.path.join(BASE_DIR, MULTI_NAME)


# ---------- HELPERS ----------
def safe_get(d, *keys):
    """Safely extract nested dictionary values."""
    for k in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
    return d


def find_data_subfolder(base_path):
    """Find subfolder containing profile_export_aiperf.json."""
    for root, _, files in os.walk(base_path):
        if 'profile_export_aiperf.json' in files:
            return root
    return None


def load_json(path):
    try:
        with open(path, 'r') as f:
            return json.load(f)
    except Exception:
        return None


def load_jsonl_normalized(path):
    try:
        df = pd.read_json(path, lines=True)
        if df.empty:
            return pd.DataFrame()

        dfs = []
        if 'metadata' in df.columns:
            dfs.append(pd.json_normalize(df['metadata']))
        if 'metrics' in df.columns:
            dfs.append(pd.json_normalize(df['metrics']))

        return pd.concat(dfs, axis=1, join='inner') if dfs else pd.DataFrame()
    except Exception:
        return pd.DataFrame()


def extract_machine_name(log_text):
    if not log_text:
        return "Unknown"

    match = re.search(r'Node\(s\)\s*:\s*([^\n]+)', log_text)
    if match:
        return match.group(1).strip()

    match = re.search(r'SubmitHost\s*:\s*([^\n]+)', log_text)
    if match:
        return match.group(1).strip()

    return "Unknown"


# ---------- CORE LOADER ----------
def load_experiment(base_path):
    name = os.path.basename(base_path)

    data_path = find_data_subfolder(base_path)
    if not data_path:
        print(f"Skipping {name}: no profile_export_aiperf.json found")
        return None

    print(f"Processing: {name}")

    # Load core files
    profile = load_json(os.path.join(data_path, 'profile_export_aiperf.json'))
    telemetry = pd.read_csv(os.path.join(data_path, 'telemetry.csv')) \
        if os.path.exists(os.path.join(data_path, 'telemetry.csv')) else None

    jsonl_df = load_jsonl_normalized(os.path.join(data_path, 'profile_export.jsonl'))

    server_metrics = load_json(os.path.join(data_path, 'server_metrics_export.json'))
    inputs = load_json(os.path.join(data_path, 'inputs.json'))

    # Logs
    job_id = name.split('_')[-1]

    err_path_1 = os.path.join(base_path, f"{job_id}.err")
    err_path_2 = os.path.join(base_path, f"{name}.err")

    out_path_1 = os.path.join(base_path, f"{job_id}.out")
    out_path_2 = os.path.join(base_path, f"{name}.out")

    err_log = None
    out_log = None

    for p in [err_path_1, err_path_2]:
        if os.path.exists(p):
            with open(p, 'r', encoding='utf-8', errors='ignore') as f:
                err_log = f.read()
            break

    for p in [out_path_1, out_path_2]:
        if os.path.exists(p):
            with open(p, 'r', encoding='utf-8', errors='ignore') as f:
                out_log = f.read()
            break

    machine = extract_machine_name(out_log)

    return {
        'experiment_name': name,
        'profile': profile,
        'telemetry': telemetry,
        'jsonl': jsonl_df,
        'server_metrics': server_metrics,
        'inputs': inputs,
        'err': err_log,
        'out': out_log,
        'machine': machine
    }


# ---------- METRIC EXTRACTION ----------
def extract_metrics(exp):
    profile = exp['profile']
    df_jsonl = exp['jsonl']
    telemetry = exp['telemetry']

    if not profile:
        return None

    def m(name, field):
        return safe_get(profile, name, field)

    metrics = {
        'experiment_name': exp['experiment_name'],
        'machine_name': exp['machine'],

        # -------- Core metrics --------
        'request_throughput_avg': m('request_throughput', 'avg'),

        'request_latency_avg_ms': m('request_latency', 'avg'),
        'request_latency_std_ms': m('request_latency', 'std'),

        'time_to_first_token_avg_ms': m('time_to_first_token', 'avg'),
        'time_to_first_token_std_ms': m('time_to_first_token', 'std'),

        'inter_token_latency_avg_ms': m('inter_token_latency', 'avg'),
        'inter_token_latency_std_ms': m('inter_token_latency', 'std'),

        'time_to_second_token_avg_ms': m('time_to_second_token', 'avg'),
        'time_to_second_token_std_ms': m('time_to_second_token', 'std'),

        'output_token_throughput_avg': m('output_token_throughput_per_user', 'avg'),
        'output_token_throughput_std': m('output_token_throughput_per_user', 'std'),
    }

    # -------- Percentiles (expanded) --------
    if df_jsonl is not None and not df_jsonl.empty:

        def add_percentiles(col_name, prefix):
            if col_name in df_jsonl.columns:
                metrics[f'{prefix}_p50'] = df_jsonl[col_name].quantile(0.50)
                metrics[f'{prefix}_p90'] = df_jsonl[col_name].quantile(0.90)
                metrics[f'{prefix}_p99'] = df_jsonl[col_name].quantile(0.99)

        # Request-level
        add_percentiles('request_latency.value', 'request_latency_ms')

        # TTFT (time to first token)
        add_percentiles('time_to_first_token.value', 'ttft_ms')

        # Time to second token
        add_percentiles('time_to_second_token.value', 'ttst_ms')

        # Inter-token latency (token generation speed)
        add_percentiles('inter_token_latency.value', 'inter_token_latency_ms')

        # Optional: time per output token (if present)
        add_percentiles('time_per_output_token.value', 'time_per_output_token_ms')

        # Optional: total output token generation time (if exists)
        add_percentiles('output_token_time.value', 'output_token_time_ms')

    # -------- Input / Output lengths --------
    if df_jsonl is not None and 'input_sequence_length.value' in df_jsonl:
        metrics['input_sequence_length_avg'] = df_jsonl['input_sequence_length.value'].mean()
        metrics['input_sequence_length_std'] = df_jsonl['input_sequence_length.value'].std()

    if df_jsonl is not None and 'output_sequence_length.value' in df_jsonl:
        metrics['output_sequence_length_avg'] = df_jsonl['output_sequence_length.value'].mean()
        metrics['output_sequence_length_std'] = df_jsonl['output_sequence_length.value'].std()
    else:
        metrics['output_sequence_length_avg'] = m('output_sequence_length', 'avg')
        metrics['output_sequence_length_std'] = m('output_sequence_length', 'std')

    # -------- Concurrency (Little's Law) --------
    if metrics['request_throughput_avg'] and metrics['request_latency_avg_ms']:
        metrics['avg_concurrency'] = (
            metrics['request_throughput_avg'] *
            metrics['request_latency_avg_ms'] / 1000.0
        )
    else:
        metrics['avg_concurrency'] = None

    return metrics


# ---------- PIPELINE ----------
results = []

# Multi experiments
if os.path.exists(MULTI_PATH):
    for entry in os.scandir(MULTI_PATH):
        if entry.is_dir():
            exp = load_experiment(entry.path)
            if exp:
                metrics = extract_metrics(exp)
                if metrics:
                    results.append(metrics)

# Single experiment
if os.path.exists(SINGLE_PATH):
    exp = load_experiment(SINGLE_PATH)
    if exp:
        metrics = extract_metrics(exp)
        if metrics:
            results.append(metrics)

# ---------- FINAL DATAFRAME ----------
df_multi_experiment_summary = pd.DataFrame(results)

if not df_multi_experiment_summary.empty:
    print("\nSummary:")
    display(df_multi_experiment_summary)
else:
    print("No valid data found.")

Processing: N2_poti_PP_long_r1_780376
Processing: N2_poti_TP_short_r1_780378
Processing: N2_poti_PP_short_r1_780413
Processing: N1_tupi_none_short_r1_780423
Processing: N2_poti_TP_long_r1_780377
Processing: N1_tupi_none_long_r1_780424
Processing: N1_tupi_none_short_r1_778989

Summary:


,experiment_name,machine_name,request_throughput_avg,request_latency_avg_ms,request_latency_std_ms,time_to_first_token_avg_ms,time_to_first_token_std_ms,inter_token_latency_avg_ms,inter_token_latency_std_ms,time_to_second_token_avg_ms,...,ttst_ms_p90,ttst_ms_p99,inter_token_latency_ms_p50,inter_token_latency_ms_p90,inter_token_latency_ms_p99,input_sequence_length_avg,input_sequence_length_std,output_sequence_length_avg,output_sequence_length_std,avg_concurrency
0,N2_poti_PP_long_r1_780376,"poti[1,5]",0.056931,17561.504580,7.605487,397.261167,0.468119,33.888200,0.291482,33.458546,...,33.662829,33.726960,33.937505,34.155432,34.615519,1024.0,0.0,507.533333,4.400104,0.999788
1,N2_poti_TP_short_r1_780378,"poti[1,5]",0.241817,4133.885852,12.290393,555.136192,0.780071,28.186573,0.102223,28.177835,...,28.488397,29.108162,28.127366,28.330452,28.360484,128.0,0.0,127.966667,0.182574,0.999643
2,N2_poti_PP_short_r1_780413,"poti[1,3]",0.231840,4311.656248,2.844926,78.803583,0.256412,33.320868,0.051373,33.196647,...,33.329718,33.402316,33.326377,33.357196,33.380847,128.0,0.0,128.033333,0.182574,0.999613
3,N1_tupi_none_short_r1_780423,tupi6,0.471978,2117.053485,0.707969,25.959997,0.313330,16.478575,0.070244,16.331817,...,16.420658,16.480939,16.465979,16.473827,16.745563,128.0,0.0,127.900000,0.547723,0.999203
4,N2_poti_TP_long_r1_780377,"poti[1,5]",0.054236,18434.523031,51.408432,3861.769976,3.906470,28.783393,0.360213,28.279900,...,28.561795,28.696231,28.792462,29.037345,29.962845,1024.0,0.0,507.366667,6.445974,0.999823
5,N1_tupi_none_long_r1_780424,tupi6,0.116210,8602.588876,3.250540,105.446350,0.565858,16.763712,0.112910,16.366283,...,16.460273,16.870995,16.820406,16.862317,17.016969,1024.0,0.0,507.900000,3.457625,0.999703
6,N1_tupi_none_short_r1_778989,Unknown,0.460778,2167.183647,1.555006,30.173936,0.825552,16.883306,0.247729,16.534861,...,16.570001,20.286288,16.826017,16.832176,17.886813,128.0,0.0,127.600000,1.714039,0.998591


### Create Summarised CSV

In [9]:
df_multi_experiment_summary.to_csv("summary.csv", index=False)